### RaschPy conditional Rasch analysis: worked example (SLM)

A conditional analysis in the style of Christensen, Kreiner & Mesbah (2013): every step conditions on each person's total score, so the person parameters drop out. The steps are CML estimation, overall fit, item fit, local dependence and dimensionality, DIF, person fit and targeting.

#### Data

This example uses the Abbreviated Mental Test Score (AMTS) data from **iarm**, an R package for Rasch analysis (Mueller, 2022; https://CRAN.R-project.org/package=iarm). The data are 197 patients' responses to 10 items scored 0 (incorrect) or 1 (correct), with each patient's age group and sex. One patient has a missing response. To run the notebook, export the data once from R into the working directory:

```r
install.packages("iarm")
write.csv(iarm::amts, "amts.csv", row.names = FALSE)
```

In [1]:
import os
import numpy as np
import pandas as pd
from raschpy import SLM

os.chdir('my_working_directory')

amts = pd.read_csv('amts.csv')
items = list(amts.columns[3:])
responses = amts[items]
exogenous = amts[['agegrp', 'sex']]

To reproduce iarm's results exactly, keep only the 196 patients who answered every item:

In [2]:
complete = responses.notna().all(axis=1)
responses = responses[complete]
exogenous = exogenous[complete]

#### 1. CML estimation

`estimation='cml'` gives conditional maximum likelihood estimates. The estimates, infit and outfit, item-restscore, Andersen and Martin-Löf results agree with iarm's to the decimals it prints.

In [3]:
slm = SLM(responses,
          exogenous=exogenous)
slm.calibrate(estimation='cml')
slm.std_errors()  # sandwich standard errors

round(pd.DataFrame({'Location': slm.items, 'SE': slm.item_se}), 3)

,Location,SE
age,-0.618,0.198
time,0.053,0.197
address,2.039,0.204
name,-0.618,0.202
year,0.135,0.182
dob,-1.752,0.279
month,0.373,0.154
firstww,-0.158,0.218
monarch,0.175,0.191
countbac,0.373,0.218


Standard errors from the inverse Hessian instead, as most CML software reports them:

In [4]:
slm.std_errors(se_method='hessian')
hessian_se = slm.item_se
slm.std_errors()

round(pd.DataFrame({'Sandwich': slm.item_se, 'Hessian': hessian_se}), 3)

,Sandwich,Hessian
age,0.198,0.210
time,0.197,0.194
address,0.204,0.191
name,0.202,0.210
year,0.182,0.193
dob,0.279,0.264
month,0.154,0.189
firstww,0.218,0.198
monarch,0.191,0.192
countbac,0.218,0.189


#### 2. Overall fit

Andersen's conditional likelihood ratio test, splitting the persons by total score, by sex and by age group:

In [5]:
slm.global_fit(method='andersen',
               split='score',
               seed=1)
slm.global_fit_summary

/Users/markelliott/Documents/RaschPy-dev/raschpy/base.py:7658: UserWarning: 2 item(s) left out of the likelihood ratio test because they have no CML estimates in one group (not responded to, an unused category, or not compared with the other items in both directions): ['age', 'dob']. See global_fit_excluded.
  out = self._global_fit_run(


method                                                                  andersen
split                                                                      score
statistic                                                              13.942466
df                                                                             7
split_prop                                                                   0.5
log-likelihoods                {'low': -284.968390509025, 'high': -89.7789663...
p_method                                                                    chi2
p-value                                                                 0.052214
p-value (chi-square)                                                    0.052214
Monte Carlo samples                                                         None
Monte Carlo samples skipped                                                 None
Monte Carlo stopped early                                                   None
reject                      

In [6]:
slm.global_fit(method='andersen',
               split='exogenous',
               covariate='sex',
               seed=1)
slm.global_fit_summary

method                                                                  andersen
split                                                                  exogenous
statistic                                                              18.753399
df                                                                             9
covariate                                                                    sex
log-likelihoods                {'female': -316.73122852221275, 'male': -144.7...
p_method                                                                    chi2
p-value                                                                 0.027373
p-value (chi-square)                                                    0.027373
Monte Carlo samples                                                         None
Monte Carlo samples skipped                                                 None
Monte Carlo stopped early                                                   None
reject                      

In [7]:
slm.global_fit(method='andersen',
               split='exogenous',
               covariate='agegrp',
               seed=1)
slm.global_fit_summary

method                                                                  andersen
split                                                                  exogenous
statistic                                                              18.558363
df                                                                            18
covariate                                                                 agegrp
log-likelihoods                {'66-75': -32.9243746616178, '76-85': -219.918...
p_method                                                                    chi2
p-value                                                                 0.419479
p-value (chi-square)                                                    0.419479
Monte Carlo samples                                                         None
Monte Carlo samples skipped                                                 None
Monte Carlo stopped early                                                   None
reject                      

Martin-Löf's test of unidimensionality, splitting the items at their median location. Most cells of its score table are sparse here, so the Monte Carlo *p*-value is used:

In [8]:
slm.global_fit(method='martin-lof',
               p_method='monte-carlo',
               seed=1)
slm.global_fit_summary

method                                                                 martin-lof
split                                                                      median
statistic                                                               21.060059
df                                                                             24
subsets                         {'A': ['age', 'time', 'name', 'dob', 'firstww'...
log-likelihoods                 {'A': -154.54741130122073, 'B': -193.397432227...
p_method                                                              monte-carlo
p-value                                                                  0.469062
p-value (chi-square)                                                      0.63518
Monte Carlo samples                                                           500
Monte Carlo samples skipped                                                     0
Monte Carlo stopped early                                                   False
reject          

#### 3. Item fit

Conditional infit and outfit, item-restscore and the conditional item chi-square Q1, with *p*-values and cutoffs from 200 datasets simulated from the model given each person's total, and Westfall–Young familywise *p*-values:

In [9]:
slm.fit_statistics(conditional=True,
                   bootstrap_samples=200,
                   seed=1)
slm.item_stats_df(full=True)

,Estimate,SE,2.5%,97.5%,Count,Facility,Infit MS,Infit Z,Outfit MS,Outfit Z,...,PM corr,Exp PM corr,Observed gamma,Expected gamma,Restscore p (BH),Q1,Q1 df,Q1 p (BH),Infit p (Westfall-Young),Outfit p (Westfall-Young)
age,-0.618,0.198,-1.009,-0.228,196,0.786,0.868,-1.051,0.641,-1.464,...,0.616,0.482,0.838,0.751,0.236,1.673,1,0.392,0.791,0.632
time,0.053,0.197,-0.337,0.442,196,0.709,1.030,0.311,1.102,0.581,...,0.569,0.499,0.709,0.737,0.597,0.007,1,0.931,0.940,0.940
address,2.039,0.204,1.636,2.442,196,0.423,1.120,1.294,1.145,0.586,...,0.423,0.370,0.616,0.687,0.234,0.881,1,0.580,0.552,0.940
name,-0.618,0.202,-1.016,-0.221,196,0.786,0.915,-0.651,0.811,-0.670,...,0.587,0.482,0.797,0.751,0.597,0.587,1,0.634,0.826,0.920
year,0.135,0.182,-0.224,0.493,196,0.699,0.881,-1.129,0.716,-1.656,...,0.651,0.500,0.819,0.735,0.236,1.676,1,0.392,0.816,0.632
dob,-1.752,0.279,-2.302,-1.202,196,0.883,1.092,0.554,0.820,-0.225,...,0.381,0.403,0.788,0.781,0.982,0.024,1,0.931,0.826,0.940
month,0.373,0.154,0.069,0.676,196,0.668,0.677,-3.497,0.590,-2.770,...,0.745,0.498,0.884,0.730,0.025,10.101,1,0.008,0.015,0.114
firstww,-0.158,0.218,-0.588,0.271,196,0.735,1.202,1.726,1.328,1.490,...,0.475,0.497,0.617,0.741,0.061,9.305,1,0.008,0.358,0.592
monarch,0.175,0.191,-0.201,0.551,196,0.694,0.964,-0.309,0.888,-0.579,...,0.611,0.500,0.774,0.734,0.597,0.357,1,0.687,0.940,0.940
countbac,0.373,0.218,-0.057,0.802,196,0.668,1.310,2.762,1.446,2.350,...,0.446,0.498,0.587,0.730,0.025,9.766,1,0.008,0.020,0.075


In [10]:
round(slm.item_restscore_table, 3)

,Observed gamma,Expected gamma,SE,Simulated mean,Simulated SD,z,p,p (BH),p (bootstrap),"p (bootstrap, Westfall-Young)"
age,0.838,0.751,0.044,0.756,0.056,1.470,0.142,0.236,0.144,0.567
time,0.709,0.737,0.066,0.741,0.053,-0.617,0.537,0.597,0.577,0.896
address,0.616,0.687,0.071,0.693,0.046,-1.677,0.094,0.234,0.109,0.502
name,0.797,0.751,0.054,0.756,0.058,0.717,0.473,0.597,0.418,0.876
year,0.819,0.735,0.042,0.738,0.053,1.538,0.124,0.236,0.164,0.582
dob,0.788,0.781,0.064,0.790,0.064,-0.022,0.982,0.982,0.995,0.995
month,0.884,0.730,0.038,0.736,0.052,2.859,0.004,0.025,0.005,0.040
firstww,0.617,0.741,0.078,0.744,0.054,-2.360,0.018,0.061,0.025,0.154
monarch,0.774,0.734,0.051,0.740,0.054,0.619,0.536,0.597,0.507,0.896
countbac,0.587,0.730,0.074,0.734,0.052,-2.808,0.005,0.025,0.005,0.060


In [11]:
round(slm.item_q1_table, 3)

,Q1,df,p,p (BH),p (bootstrap),"p (bootstrap, Westfall-Young)"
age,1.673,1,0.196,0.392,0.179,0.756
time,0.007,1,0.931,0.931,0.955,0.990
address,0.881,1,0.348,0.580,0.328,0.841
name,0.587,1,0.444,0.634,0.517,0.945
year,1.676,1,0.195,0.392,0.249,0.841
dob,0.024,1,0.877,0.931,0.856,0.990
month,10.101,1,0.001,0.008,0.005,0.025
firstww,9.305,1,0.002,0.008,0.005,0.025
monarch,0.357,1,0.550,0.687,0.547,0.945
countbac,9.766,1,0.002,0.008,0.005,0.020


In [12]:
round(slm.item_fit_bootstrap, 3)

,Infit MS,Infit 95% low,Infit 95% high,Infit p,Infit p (Westfall-Young),Outfit MS,Outfit 95% low,Outfit 95% high,Outfit p,Outfit p (Westfall-Young)
age,0.868,0.811,1.212,0.244,0.791,0.641,0.553,1.448,0.144,0.632
time,1.030,0.811,1.180,0.796,0.940,1.102,0.653,1.313,0.602,0.940
address,1.120,0.873,1.163,0.104,0.552,1.145,0.666,1.538,0.418,0.940
name,0.915,0.794,1.191,0.383,0.826,0.811,0.574,1.455,0.423,0.920
year,0.881,0.776,1.199,0.294,0.816,0.716,0.644,1.363,0.164,0.632
dob,1.092,0.792,1.242,0.388,0.826,0.820,0.350,1.811,0.756,0.940
month,0.677,0.826,1.194,0.005,0.015,0.590,0.688,1.261,0.025,0.114
firstww,1.202,0.807,1.187,0.055,0.358,1.328,0.665,1.408,0.100,0.592
monarch,0.964,0.829,1.193,0.766,0.940,0.888,0.667,1.323,0.552,0.940
countbac,1.310,0.810,1.181,0.005,0.020,1.446,0.696,1.291,0.010,0.075


#### 4. Local dependence and dimensionality

Q3\* (residual correlations minus their mean), with the simulated cutoff, and the first contrast of the residuals' principal components:

In [13]:
slm.res_corr_analysis()
round(slm.q3_star, 3)

,age,time,address,name,year,dob,month,firstww,monarch,countbac
age,1.109,0.121,0.085,-0.054,0.149,-0.090,0.113,-0.086,-0.046,-0.082
time,0.121,1.109,-0.069,-0.056,0.106,0.015,0.183,0.041,-0.237,-0.109
address,0.085,-0.069,1.109,0.008,-0.049,-0.028,-0.041,0.028,0.037,-0.104
name,-0.054,-0.056,0.008,1.109,0.015,0.133,0.027,-0.103,0.149,-0.005
year,0.149,0.106,-0.049,0.015,1.109,-0.024,-0.209,-0.057,-0.046,0.092
dob,-0.090,0.015,-0.028,0.133,-0.024,1.109,0.185,0.087,-0.063,0.009
month,0.113,0.183,-0.041,0.027,-0.209,0.185,1.109,-0.084,-0.002,0.001
firstww,-0.086,0.041,0.028,-0.103,-0.057,0.087,-0.084,1.109,0.105,-0.078
monarch,-0.046,-0.237,0.037,0.149,-0.046,-0.063,-0.002,0.105,1.109,0.034
countbac,-0.082,-0.109,-0.104,-0.005,0.092,0.009,0.001,-0.078,0.034,1.109


In [14]:
round(slm.q3_bootstrap, 3)

,Item 1,Item 2,Q3,Expected Q3,95% cutoff,p,p (Westfall-Young)
0,dob,month,0.075,-0.068,0.055,0.040,0.896
1,time,month,0.074,-0.126,0.008,0.010,0.388
2,age,year,0.040,-0.109,0.048,0.065,0.920
3,name,monarch,0.040,-0.112,0.036,0.055,0.920
4,name,dob,0.024,-0.105,0.055,0.085,0.990
5,age,time,0.011,-0.111,0.067,0.100,0.990
6,age,month,0.004,-0.107,0.038,0.109,0.990
7,time,year,-0.004,-0.114,0.019,0.095,0.990
8,firstww,monarch,-0.004,-0.113,0.041,0.104,0.990
9,year,countbac,-0.017,-0.132,0.005,0.100,0.990


In [15]:
slm.residual_contrast_bootstrap

{'first_contrast': 1.4800607619662085,
 'expected': 1.4622680403351223,
 'cutoff_95': 1.5710176444595079,
 'p': 0.38308457711442784}

Partial gamma tests of local dependence (Kreiner & Christensen, 2004):

In [16]:
round(slm.local_dependence_gamma(), 3)

,Item,Paired item,Gamma,z,p,p (BH),Flagged
0,dob,month,0.857,1.777,0.076,0.645,False
1,time,month,0.835,3.366,0.001,0.069,False
2,address,age,0.568,1.515,0.130,0.687,False
3,month,dob,0.560,1.223,0.221,0.702,False
4,countbac,year,0.459,2.109,0.035,0.628,False
...,...,...,...,...,...,...,...
85,time,countbac,-0.550,-2.200,0.028,0.628,False
86,monarch,time,-0.570,-2.191,0.028,0.628,False
87,dob,address,-0.651,-1.466,0.143,0.687,False
88,year,month,-0.662,-1.643,0.100,0.645,False


#### 5. Differential item functioning

DIF by sex, with the conditional likelihood ratio omnibus test:

In [17]:
slm.dif_test('sex',
             seed=1)
round(slm.dif_table, 3)

,Group,Reference,Focal,Focal (purified),Difference,SE,t,p,p (corrected),Selected,Flagged,df
Item,,,,,,,,,,,,
age,male,-0.379,-1.243,-1.501,-1.123,0.562,-1.998,0.050,0.248,False,False,68.737
time,male,-0.116,0.429,0.310,0.427,0.486,0.878,0.382,0.492,True,False,89.787
address,male,2.091,2.023,1.724,-0.367,0.532,-0.690,0.492,0.492,True,False,86.369
name,male,-0.586,-0.672,-0.974,-0.388,0.495,-0.783,0.436,0.492,True,False,92.684
year,male,0.252,-0.077,-0.422,-0.674,0.477,-1.413,0.161,0.386,True,False,83.216
dob,male,-2.092,-1.034,-1.234,0.858,0.655,1.310,0.193,0.386,True,False,96.364
month,male,0.193,0.776,0.726,0.534,0.374,1.428,0.157,0.386,True,False,96.240
firstww,male,0.193,-1.034,-1.292,-1.485,0.581,-2.554,0.013,0.126,False,False,79.158
monarch,male,0.252,0.056,-0.260,-0.512,0.513,-0.998,0.321,0.492,True,False,75.539


In [18]:
slm.dif_omnibus_table

,LR,df,p,Flagged
male,18.753399,9,0.027373,True


#### 6. Person fit

Each person's conditional *p*-value: the probability, given their total score, of a response pattern as unlikely as theirs:

In [19]:
p = slm.person_conditional_p
print(f"{(p < 0.05).sum()} of {p.notna().sum()} persons have p < 0.05")
round(p.sort_values().head(10), 4)

4 of 145 persons have p < 0.05


50     0.0020
91     0.0070
114    0.0080
17     0.0220
178    0.0509
169    0.0579
85     0.0659
98     0.0799
147    0.0889
11     0.0999
Name: Conditional p, dtype: float64

#### 7. Targeting and reliability

In [20]:
slm.targeting()
slm.targeting_summary

Population mean                   1.527681
Population SD                     2.091625
Target                           -0.040239
Target - population mean         -1.567921
Maximum information               2.161907
Expected information              1.200661
Expected / maximum information    0.555371
Test difficulty                  -0.014743
Expected |bias|                   0.292733
Expected SEM                      0.745752
Expected RMSE                     0.925730
rho1                              0.855610
rho2                              0.855610
rho3                              0.827810
rho4                              0.827810
rho5                              0.884624
rho6                              0.784518
Name: Targeting, dtype: float64

#### 8. Summary

The results above in one table:

In [21]:
slm.global_stats_df()

,Section,Value,df,p,Details
Statistic,,,,,
Overall Q1 (item scores),Overall fit,30.940,NaN,0.010,"2 ability groups; Monte Carlo p, 99 samples"
Andersen LR,Overall fit,13.942,7.0,0.052,split=score
Andersen LR,Overall fit,18.753,9.0,0.027,split=exogenous; covariate=sex
Andersen LR,Overall fit,18.558,18.0,0.419,split=exogenous; covariate=agegrp
Martin-Lof,Dimensionality and local dependence,21.060,24.0,0.469,split=median; p_method=monte-carlo; 61% of sco...
First residual contrast,Dimensionality and local dependence,1.480,NaN,0.383,bootstrap 95% cutoff 1.571
Q3* (largest),Dimensionality and local dependence,0.185,NaN,NaN,conditional residuals
Item pairs with local dependence,Dimensionality and local dependence,0.000,NaN,NaN,"of 90 ordered pairs; partial gamma, BH p <= 0...."
Items flagged by Q1,Item fit,3.000,NaN,NaN,of 10; BH p <= 0.05


#### References

Christensen, K. B., Kreiner, S. & Mesbah, M. (Eds.) (2013). *Rasch Models in Health*. Wiley.

Kreiner, S. & Christensen, K. B. (2004). Analysis of local dependence and multidimensionality in graphical loglinear Rasch models. *Communications in Statistics – Theory and Methods*, 33, 1239–1276.

Mueller, M. (2022). *iarm: Item Analysis in Rasch Models*. R package version 0.4.3. https://CRAN.R-project.org/package=iarm